# NFL Week 1 Analysis — Data Preparation

This notebook loads, cleans, validates, and prepares the NFL data used by the analysis notebooks.

## Data sources

- **NFL team statistics:** `nflreadpy`
- **Betting and game results:** Kaggle's `nfl-scores-and-betting-data` dataset

The analysis covers **regular-season games from 2006–2024**.

The prepared datasets produced by this notebook are saved as Parquet files in `../data/` for use by the downstream analysis notebooks.

## 1. Load NFL team statistics

The NFL team statistics are loaded at the **team-game level**, meaning each game normally contributes one row for each team.

In [2]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [3]:
team_stats_full = nfl.load_team_stats(
    seasons=list(range(2006, 2025)),
    summary_level="week"
)

### Select relevant columns

Only the variables needed for the project are retained. Keeping a defined set of columns makes the downstream dataset easier to understand and reduces unnecessary data.

In [4]:
keep_col = [
    # Identifiers
    "season",
    "week",
    "team",
    "season_type",
    "game_id",
    "opponent_team",

    # Passing
    "completions",
    "attempts",
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "sacks_suffered",
    "passing_air_yards",
    "passing_yards_after_catch",
    "passing_first_downs",
    "passing_epa",
    "passing_cpoe",

    # Rushing
    "carries",
    "rushing_yards",
    "rushing_tds",
    "rushing_first_downs",
    "rushing_epa",

    # Defense
    "def_tackles_solo",
    "def_tackles_with_assist",
    "def_tackle_assists",
    "def_tackles_for_loss",
    "def_fumbles_forced",
    "def_sacks",
    "def_sack_yards",
    "def_qb_hits",
    "def_interceptions",
    "def_pass_defended",
    "def_tds",
    "def_fumbles",
    "def_safeties",

    # Turnovers / penalties
    "fumbles_total",
    "fumbles_lost_total",
    "penalties",
    "penalty_yards",

    # Special teams
    "fg_made",
    "fg_att",
    "fg_missed",
    "fg_blocked",
    "fg_long",
    "fg_pct",
    "punt_returns",
    "punt_return_yards",
    "kickoff_returns",
    "kickoff_return_yards"
]

team_stats_w_post = team_stats_full.select(keep_col).to_pandas()

team_stats = team_stats_w_post[
    team_stats_w_post["season_type"] == "REG"
].copy()

### Validate NFL team statistics

The expected structure is two team-level observations per game.

In [5]:
print("Shape:", team_stats.shape)
print("Seasons:", team_stats["season"].min(), "to", team_stats["season"].max())
print("Unique games:", team_stats["game_id"].nunique())

rows_per_game = (
    team_stats
    .groupby("game_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nRows per game:")
print(rows_per_game)

print("\nMissing values in selected columns:")
print(
    team_stats
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

Shape: (9854, 49)
Seasons: 2006 to 2024
Unique games: 4927

Rows per game:
2    4927
Name: count, dtype: int64

Missing values in selected columns:
fg_long          1812
fg_pct           1261
passing_cpoe       11
season              0
week                0
opponent_team       0
completions         0
season_type         0
team                0
passing_yards       0
dtype: int64


## 2. Load and filter betting data

The Kaggle dataset contains game results, betting lines, team names, and other game-level information.

Only regular-season games from 2006–2024 are retained so that the period matches the NFL team statistics.

In [6]:
betting = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "tobycrabtree/nfl-scores-and-betting-data",
    "spreadspoke_scores.csv"
)

betting = betting[
    (betting["schedule_season"] >= 2006) &
    (betting["schedule_season"] <= 2024) &
    (betting["schedule_playoff"] == False)
].copy()

print("Shape:", betting.shape)
print("Unique seasons:", betting["schedule_season"].nunique())
print("Unique games:", len(betting))

Shape: (4927, 17)
Unique seasons: 19
Unique games: 4927


## 3. Standardize team names

The two data sources use different team-name conventions. Betting data is converted to the abbreviations used by `nflreadpy`.

Historical franchise names are also mapped to their modern abbreviations.

In [7]:
team_name_map = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Las Vegas Raiders": "LV",
    "Los Angeles Chargers": "LAC",
    "Los Angeles Rams": "LA",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Oakland Raiders": "LV",
    "San Diego Chargers": "LAC",
    "St. Louis Rams": "LA",
    "Washington Redskins": "WAS",
    "Washington Football Team": "WAS",
    "Washington Commanders": "WAS"
}

betting["team_home"] = betting["team_home"].map(team_name_map)
betting["team_away"] = betting["team_away"].map(team_name_map)

print("Unmapped home teams:", betting["team_home"].isna().sum())
print("Unmapped away teams:", betting["team_away"].isna().sum())

Unmapped home teams: 0
Unmapped away teams: 0


## 4. Construct and validate game IDs

A common game ID is constructed from:

`season_week_away_team_home_team`

Historical team abbreviations in the `nflreadpy` IDs are then normalized so that the two sources can be matched.

In [8]:
betting["game_id"] = (
    betting["schedule_season"].astype(str)
    + "_"
    + betting["schedule_week"].astype(str).str.zfill(2)
    + "_"
    + betting["team_away"]
    + "_"
    + betting["team_home"]
)

game_id_map = {
    "STL": "LA",
    "SD": "LAC",
    "OAK": "LV"
}

team_stats["game_id_normalized"] = (
    team_stats["game_id"]
    .str.split("_")
    .apply(
        lambda x: "_".join([
            x[0],
            x[1],
            game_id_map.get(x[2], x[2]),
            game_id_map.get(x[3], x[3])
        ])
    )
)

In [9]:
print("Betting games:", betting["game_id"].nunique())
print("NFL team-stat games:", team_stats["game_id_normalized"].nunique())

print(
    "Matching games:",
    betting["game_id"].isin(team_stats["game_id_normalized"]).sum()
)

print(
    "Unmatched betting games:",
    (~betting["game_id"].isin(team_stats["game_id_normalized"])).sum()
)

print(
    "Unmatched team-stat games:",
    (~team_stats["game_id_normalized"].isin(betting["game_id"])).sum()
)

Betting games: 4927
NFL team-stat games: 4927
Matching games: 4927
Unmatched betting games: 0
Unmatched team-stat games: 0


## 5. Merge betting data with team statistics

Betting information is merged onto the team-game-level NFL statistics.

A `many_to_one` validation is used because each betting game should correspond to multiple team-stat rows, while the betting dataset should contain one row per game.

In [10]:
team_stats = team_stats.merge(
    betting,
    left_on="game_id_normalized",
    right_on="game_id",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(team_stats))
print("Unique games:", team_stats["game_id_normalized"].nunique())
print(
    "Missing betting rows:",
    team_stats["team_favorite_id"].isna().sum()
)

print("\nRows per game:")
print(
    team_stats
    .groupby("game_id_normalized")
    .size()
    .value_counts()
    .sort_index()
)

Rows: 9854
Unique games: 4927
Missing betting rows: 0

Rows per game:
2    4927
Name: count, dtype: int64


## 6. Normalize favorite identifiers

The betting dataset uses identifiers such as `LAR` and `LVR`. These are normalized to match the team abbreviations used elsewhere in the project.

`PICK` represents a pick'em game with no betting favorite.

In [11]:
favorite_id_map = {
    "LAR": "LA",
    "LVR": "LV"
}

team_stats["favorite_team"] = (
    team_stats["team_favorite_id"]
    .replace(favorite_id_map)
)

print(
    "Favorite IDs:",
    sorted(team_stats["favorite_team"].dropna().unique())
)

print(
    "Pick'em games:",
    team_stats.loc[
        team_stats["favorite_team"] == "PICK",
        "game_id_normalized"
    ].nunique()
)

Favorite IDs: ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PICK', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']
Pick'em games: 20


## 7. Create the game-level dataset

Most of the project compares complete games rather than individual teams. The `games` dataset therefore contains one row per game.

In [12]:
games = (
    team_stats[
        [
            "game_id_normalized",
            "season",
            "week",
            "team_favorite_id",
            "favorite_team",
            "score_home",
            "score_away",
            "team_home",
            "team_away",
            "spread_favorite"
        ]
    ]
    .drop_duplicates("game_id_normalized")
    .copy()
)

print("Rows:", len(games))
print("Unique games:", games["game_id_normalized"].nunique())

Rows: 4927
Unique games: 4927


### Definition A — Betting-market upset

A betting-market upset occurs when the pregame favorite loses.

Pick'em games are excluded because neither team was designated as the favorite.

In [13]:
games["favorite_score"] = games.apply(
    lambda row: (
        row["score_home"]
        if row["favorite_team"] == row["team_home"]
        else row["score_away"]
    ),
    axis=1
)

games["underdog_score"] = games.apply(
    lambda row: (
        row["score_away"]
        if row["favorite_team"] == row["team_home"]
        else row["score_home"]
    ),
    axis=1
)

games["favorite_margin"] = (
    games["favorite_score"] - games["underdog_score"]
)

games["betting_upset"] = (
    (games["favorite_team"] != "PICK") &
    (games["favorite_margin"] < 0)
)

games["upset_margin"] = (
    games["favorite_margin"]
    .clip(upper=0)
    .abs()
)

games.loc[
    games["favorite_team"] == "PICK",
    "upset_margin"
] = 0

In [14]:
print("Betting-market upsets:", games["betting_upset"].sum())
print(
    "Betting-market upset rate:",
    games["betting_upset"].mean()
)
print(
    "Maximum upset margin:",
    games["upset_margin"].max()
)

Betting-market upsets: 1637
Betting-market upset rate: 0.3322508625938705
Maximum upset margin: 45


## 8. Create reusable team-level game results

This dataset records one row for each team in each game and is used later to calculate team records and subsequent-season wins without rebuilding game results.

In [15]:
home_results = games[
    [
        "season",
        "week",
        "team_home",
        "score_home",
        "score_away"
    ]
].copy()

home_results["wins"] = (
    home_results["score_home"] > home_results["score_away"]
).astype(int)

home_results = home_results.rename(
    columns={"team_home": "team"}
)

away_results = games[
    [
        "season",
        "week",
        "team_away",
        "score_away",
        "score_home"
    ]
].copy()

away_results["wins"] = (
    away_results["score_away"] > away_results["score_home"]
).astype(int)

away_results = away_results.rename(
    columns={"team_away": "team"}
)

team_results = pd.concat(
    [
        home_results[["season", "week", "team", "wins"]],
        away_results[["season", "week", "team", "wins"]]
    ],
    ignore_index=True
)

print("Rows:", len(team_results))
print("Unique teams:", team_results["team"].nunique())

Rows: 9854
Unique teams: 32


## 9. Save prepared datasets

The prepared datasets are saved as Parquet files so the downstream notebooks can load them independently.

- `games.csv` — one row per game
- `team_stats.csv` — one row per team per game
- `team_results.csv` — one row per team per game containing win results

In [16]:
from pathlib import Path

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

games.to_parquet(data_dir / "games.parquet", index=False)
team_stats.to_parquet(data_dir / "team_stats.parquet", index=False)
team_results.to_parquet(data_dir / "team_results.parquet", index=False)

print("Saved:")
print(" - games.parquet")
print(" - team_stats.parquet")
print(" - team_results.parquet")

Saved:
 - games.parquet
 - team_stats.parquet
 - team_results.parquet


## 10. Final data-quality checks

These checks confirm that the prepared datasets have the expected grain, matching games, and no missing betting merges.

In [17]:
print("=== Dataset shapes ===")
print("games:", games.shape)
print("team_stats:", team_stats.shape)
print("team_results:", team_results.shape)

print("\n=== Game uniqueness ===")
print(
    "Unique games in games:",
    games["game_id_normalized"].nunique()
)
print(
    "Rows per game in team_stats:"
)
print(
    team_stats
    .groupby("game_id_normalized")
    .size()
    .value_counts()
    .sort_index()
)

print("\n=== Missing merge keys ===")
print(
    "Missing betting rows:",
    team_stats["schedule_season"].isna().sum()
)
print(
    "Missing normalized game IDs:",
    team_stats["game_id_normalized"].isna().sum()
)

print("\n=== Week 1 ===")
print(
    "Week 1 games:",
    (games["week"] == 1).sum()
)

=== Dataset shapes ===
games: (4927, 15)
team_stats: (9854, 69)
team_results: (9854, 4)

=== Game uniqueness ===
Unique games in games: 4927
Rows per game in team_stats:
2    4927
Name: count, dtype: int64

=== Missing merge keys ===
Missing betting rows: 0
Missing normalized game IDs: 0

=== Week 1 ===
Week 1 games: 303
